In [2]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [3]:

from utils.funciones_minio import crear_cliente_minio, bajar_minio
from utils.config import MINIO_EMBEDDINGS   
EMBEDDINGS_IMAGENES = "embeddings_imagenes.parquet"

In [4]:
client = crear_cliente_minio()

In [7]:
embedding_imagenes = bajar_minio(client, MINIO_EMBEDDINGS, EMBEDDINGS_IMAGENES)

In [8]:
x = np.stack(embedding_imagenes["embedding"].values)
y = embedding_imagenes["clase"].values.to_numpy()
print(x.shape)
print(y.shape)

(172306, 2048)
(172306,)


In [10]:
# 1. Convertir etiquetas de texto a números
encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y)

# 2. Convertir a formato categórico (One-Hot)
num_classes = len(encoder.classes_)
y_categorical = tf.keras.utils.to_categorical(y_encoded, num_classes=num_classes)

# 3. Split: 80% Train, 20% Temporal (que dividiremos en Val y Test)
x_train, x_temp, y_train, y_temp = train_test_split(
    x, y_categorical, test_size=0.2, random_state=42, stratify=y_encoded
)

# 4. Split del temporal: 50% Val, 50% Test (esto da 10% y 10% del total)
x_val, x_test, y_val, y_test = train_test_split(
    x_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f"Estructura de entrenamiento: {x_train.shape}")
print(f"Clases detectadas: {encoder.classes_}")
print(f"Número de clases: {num_classes}")

Estructura de entrenamiento: (137844, 2048)
Clases detectadas: ['Banyo' 'Cocina' 'Comedor' 'Dormitorio' 'Salón']
Número de clases: 5


In [ ]:
def build_model(input_shape=(2048,), num_classes=4):
    model = models.Sequential([
        layers.Input(shape=input_shape),
        
        # Estabiliza las entradas del embedding
        layers.BatchNormalization(),
        
        # Primera capa densa potente
        layers.Dense(512, activation='relu'),
        layers.Dropout(0.4), # Un poco agresivo para evitar memorización
        
        # Segunda capa para refinar patrones
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.2),
        
        # Capa de salida para las clases detectadas
        layers.Dense(num_classes, activation='softmax')
    ])
    
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model

model = build_model(num_classes=num_classes)
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ batch_normalization             │ (None, 2048)           │         8,192 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │     1,049,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        65,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,123,460 (4.29 MB)

 Trainable params: 1,119,364 (4.27 MB)

 Non-trainable params: 4,096 (16.00 KB)

In [12]:
# Configuración de callbacks
early_stop = callbacks.EarlyStopping(
    monitor='val_loss', 
    patience=5, 
    restore_best_weights=True
)

reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss', 
    factor=0.2, 
    patience=3, 
    min_lr=0.00001
)

# Entrenamiento
history = model.fit(
    x_train, y_train,
    epochs=50, # No llegará a 50 gracias al EarlyStopping
    batch_size=128, # Tamaño grande para aprovechar los 172k datos
    validation_data=(x_val, y_val),
    callbacks=[early_stop, reduce_lr]
)

Epoch 1/50


ValueError: Arguments `target` and `output` must have the same shape. Received: target.shape=(None, 5), output.shape=(None, 4)